# MarketSim M1 - 02 Curated output and the four metrics
Team: _fill in_  |  Student: Ahmed Shawky Moubarak (70-40377)

This notebook reads the **curated output** of `run_pipeline.py` (not the raw files) and shows the four Milestone 1 metrics in pandas, each with its policy. The definitions are in Appendix A of the Dataset Description.

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")
import pandas as pd, numpy as np
from pathlib import Path
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
OUT = Path("output")
canon = pd.concat([pd.read_parquet(f) for f in sorted((OUT / "canonical_bars").glob("*.parquet"))])
rej = pd.read_parquet(OUT / "rejects.parquet")
print("canonical bars:", f"{len(canon):,}", "| rejects:", f"{len(rej):,}")
canon.head(3).T

canonical bars: 4,240,455 | rejects: 35,720


,0,1,2
bar_id,RUN-2025A|INDEX|20250102T1430,RUN-2025A|INDEX|20250102T1431,RUN-2025A|INDEX|20250102T1432
source_file,bars_1m.parquet,bars_1m.parquet,bars_1m.parquet
source_row,1377798,1377799,1377800
run_id,RUN-2025A,RUN-2025A,RUN-2025A
security_id,INDEX,INDEX,INDEX
ticker_as_traded,INDEX,INDEX,INDEX
bar_start_local,2025-01-02 09:30:00,2025-01-02 09:31:00,2025-01-02 09:32:00
bar_start_utc,2025-01-02 14:30:00+00:00,2025-01-02 14:31:00+00:00,2025-01-02 14:32:00+00:00
trading_date,2025-01-02 00:00:00,2025-01-02 00:00:00,2025-01-02 00:00:00
minute_of_session,0,1,2


### Every raw row is either in the canonical table or in the rejects table

In [2]:
received = 4_276_175   # rows in bars_1m.parquet
print(len(canon) + len(rej) == received)
print(rej.reject_reason.value_counts())
print(pd.crosstab(rej.run_id, rej.reject_reason))
print("rows with more than one applicable reason (first enum value kept):", (rej.n_reasons > 1).sum())
rej[["run_id","ticker","ts","source_file","source_row","reject_reason"]].head(5)

True
reject_reason
OUTSIDE_SESSION           16764
DUPLICATE_BAR             14913
INVALID_OHLC               1272
MISSING_REQUIRED_FIELD      850
INVALID_NUMERIC_RANGE       846
MALFORMED_TIMESTAMP         425
UNKNOWN_SYMBOL              390
UNKNOWN_RUN                 260
Name: count, dtype: int64
reject_reason  DUPLICATE_BAR  INVALID_NUMERIC_RANGE  INVALID_OHLC  MALFORMED_TIMESTAMP  MISSING_REQUIRED_FIELD  OUTSIDE_SESSION  UNKNOWN_RUN  UNKNOWN_SYMBOL
run_id                                                                                                                                                      
RUN-2025A               4992                    280           422                  142                     284            11326            0             390
RUN-2025B               4970                    284           426                  142                     284             5418            0               0
RUN-2025C               4951                    282           424      

,run_id,ticker,ts,source_file,source_row,reject_reason
0,RUN-2025A,ARVT,2025-01-06 12:13:00,bars_1m.parquet,942,INVALID_OHLC
1,RUN-2025A,ARVT,2025-01-07 11:23:00,bars_1m.parquet,1282,MISSING_REQUIRED_FIELD
2,RUN-2025A,ARVT,2025-01-13 12:26:00,bars_1m.parquet,2905,MISSING_REQUIRED_FIELD
3,RUN-2025A,ARVT,2025-01-13 15:12:00,bars_1m.parquet,3070,INVALID_OHLC
4,RUN-2025A,ARVT,2025-01-14 13:01:00,bars_1m.parquet,3329,INVALID_OHLC


## Metric 1. Bar completeness
* **Definition** accepted bars / expected bars, per run, symbol and day.
* **Expected** = minutes of the session on the calendar (390, or 210 on the early close 2025-04-17) for every symbol *listed that day* (listing_date <= day < delisting_date) plus the index, minus the announced halt minutes (GALR, 45 min on 2025-02-12, applied to every run).
* **Population / window** 3 known runs, 2025-01-02 to 2025-06-30, 123 trading days (holidays excluded).
* **Null policy** a missing bar counts as not accepted. A symbol-day with zero bars stays in the table with completeness 0 (it is not dropped).
* Accepted means: in the canonical table (after dedup, session, symbol and value checks).

In [3]:
comp = pd.read_csv(OUT / "metrics" / "completeness_by_run_symbol_day.csv")
by_run = pd.read_csv(OUT / "metrics" / "m1_completeness_by_run.csv"); display(by_run)
low = comp[comp.completeness < 0.99]
print("symbol-days below 99%:", len(low), "of", len(comp))
comp.groupby("run_id").completeness.describe().round(4)

,run_id,accepted,expected,completeness
0,RUN-2025A,1413522,1414935,0.999001
1,RUN-2025B,1413467,1414935,0.998962
2,RUN-2025C,1413466,1414935,0.998962


symbol-days below 99%: 11 of 10926


,count,mean,std,min,25%,50%,75%,max
run_id,,,,,,,,
RUN-2025A,3642.0,0.999,0.0019,0.9333,0.9974,1.0,1.0,1.0
RUN-2025B,3642.0,0.999,0.0020,0.9333,0.9974,1.0,1.0,1.0
RUN-2025C,3642.0,0.999,0.0019,0.9333,0.9974,1.0,1.0,1.0


## Metric 2. OHLC validity rate
* **Definition** bars with `low <= open, close <= high` and positive prices / bars checked.
* **Population** all rows of known runs **after removing duplicate copies**, that have all four prices (a missing price is a MISSING_REQUIRED_FIELD, not an OHLC failure). Unknown-run rows are not checked. Timestamp, symbol and session problems do not change the OHLC test, so those rows stay in.
* **Null policy** rows with a null price are outside the denominator.

In [4]:
ohlc = pd.read_csv(OUT / "metrics" / "m2_ohlc_validity_by_run.csv"); display(ohlc)

,run_id,checked,valid,rate
0,RUN-2025A,1426081,1425654,0.999701
1,RUN-2025B,1419737,1419311,0.999700
2,RUN-2025C,1414333,1413909,0.999700


## Metric 3. Minute-to-daily reconciliation against the vendor
* **Derived daily bar** from the *accepted* minute bars of the same run, security and trading day: open = first minute, high = max, low = min, close = last minute, volume = sum.
* **Match** prices within half a cent and volume exact, after putting both on the same basis: the derived bars are split-adjusted for days before the JORV split (the vendor already is), and for `FENM` the vendor volume is in lots of 100 so it is compared as `round(volume / 100)`.
* **Compared** = days present on both sides. `missing_in_vendor` / `missing_in_derived` are reported separately.
* A mismatch is labelled `MISSING_MINUTES` when the derived day has fewer bars than expected (so the volume cannot be equal), else `VALUE_DIFF`.

In [5]:
rs = pd.read_csv(OUT / "metrics" / "m3_reconciliation_by_run.csv"); display(rs)
rec = pd.read_parquet(OUT / "reconciliation.parquet")
print(rec.status.value_counts().to_dict())
print(rec[rec.status == "mismatched"].mismatch_cause.value_counts().to_dict())
share = (rec.query("status == 'mismatched'").mismatch_cause == "MISSING_MINUTES").mean()
print(f"{share:.1%} of mismatches are days where minutes are missing")
rec[rec.status == "mismatched"].query("mismatch_cause == 'VALUE_DIFF'").head(6)[["run_id","security_id","trading_date","mismatch_fields","close","close_v"]]

,run_id,compared,matched,mismatched,missing_in_vendor,missing_in_derived,match_rate
0,RUN-2025A,3612,2448,1164,30,0,0.677741
1,RUN-2025B,3605,2415,1190,37,0,0.669903
2,RUN-2025C,3614,2413,1201,28,0,0.667681


{'matched': 7276, 'mismatched': 3555, 'missing_in_vendor': 95}
{'MISSING_MINUTES': 3506, 'VALUE_DIFF': 49}
98.6% of mismatches are days where minutes are missing


,run_id,security_id,trading_date,mismatch_fields,close,close_v
225,RUN-2025A,SEC001,2025-05-30,close,115.31,115.36
940,RUN-2025A,SEC007,2025-04-28,close,13.19,13.14
1539,RUN-2025A,SEC012,2025-04-03,close,84.47,84.46
1864,RUN-2025A,SEC015,2025-01-30,close,110.70,110.69
1933,RUN-2025A,SEC015,2025-05-09,close,78.14,78.09
1943,RUN-2025A,SEC015,2025-05-23,close,67.23,67.28


## Metric 4. Rejected-record and late-record rate
* **Rejected rate** = rejected / received (received = every row of bars_1m.parquet of that run).
* **Late rate** = late-accepted / accepted, where late = `received_at` more than 60 s after the end of the bar. Late bars are **accepted and flagged** `LATE` in `quality_flags`; the reason `LATE_BEYOND_WATERMARK` belongs to the Milestone 2 stream (it needs a watermark), so it is never used here.
* `RUN-TEST` has no accepted bars, so its late rate is undefined (null), not 0.

In [6]:
rl = pd.read_csv(OUT / "metrics" / "m4_reject_late_by_run.csv"); display(rl)
tot = rl[rl.run_id != "RUN-TEST"][["received","rejected","accepted","late_accepted"]].sum()
print("known runs only: reject rate", round(tot.rejected / tot.received, 5), "| late rate", round(tot.late_accepted / tot.accepted, 5))

,run_id,received,rejected,accepted,late_accepted,reject_rate,late_rate
0,RUN-2025A,1431358,17836,1413522,7076,0.012461,0.005006
1,RUN-2025B,1424991,11524,1413467,7059,0.008087,0.004994
2,RUN-2025C,1419566,6100,1413466,7064,0.004297,0.004998
3,RUN-TEST,260,260,0,0,1.000000,NaN


known runs only: reject rate 0.00829 | late rate 0.005
